In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from nptdms import TdmsFile


In [ ]:


# ── Polarisation index mapping ────────────────────────────────────────────────
_orientations = ["90", "45", "135", "0"]

def get_pol_ind(listpol):
    return [_orientations.index(p) for p in listpol]


In [ ]:


# ── File list — edit this ─────────────────────────────────────────────────────
# These files are used together to calibrate a single gain vector a = [a_90, a_45, a_135, a_0].
# Use a diverse set of files (different beads / conditions) so the calibration
# is not biased by a single recording.

tdms_file_list = [
    Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed5.tdms"),
    Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed6.tdms"),
    Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed7.tdms"),
    Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed8.tdms"),
    Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed9.tdms"),
    Path(r"C:\Users\denny\workspace\pyqtrod\data\2024_08_01_fixed_gold\fixed\fixed10.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2022_06_14_5_strept0.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2022_06_14_5_strept3.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free3.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free6.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\2free7.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free4.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free444444452.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free4444450.tdms"),
    #Path(r"C:\Users\denny\workspace\pyqtrod\data\2023_06_23_free_gold\free444449.tdms"),
]

# ── Fit settings ───────────────────────────────────────────────────────────────
fit_start_s    = None    # s — start of time window for fitting (None = beginning)
fit_end_s      = None    # s — end of time window for fitting   (None = full file)
dec_fit        = 1000    # decimation applied to each file during the fit
dec_plot       = 1000    # decimation for verification plots

# ── Check paths ───────────────────────────────────────────────────────────────
print(f"{'File':<52} {'exists':>6}")
print("-" * 60)
for p in tdms_file_list:
    exists = "YES" if p.exists() else "NO !"
    print(f"  {p.name:<50} {exists:>6}")
print(f"\nTotal: {len(tdms_file_list)} file(s) defined.")


In [ ]:


# ── Load raw channel intensities from all TDMS files ─────────────────────────

raw_data_all = {}

for tdms_path in tdms_file_list:
    if not tdms_path.exists():
        print(f"FILE NOT FOUND — skipping: {tdms_path.name}")
        continue
    try:
        tdms_obj = TdmsFile.open(str(tdms_path))
    except Exception as e:
        print(f"ERROR opening {tdms_path.name}: {e} — skipping")
        continue

    group    = tdms_obj.groups()[0]
    channels = group.channels()
    datasize = len(channels[0])

    try:
        time_inc = float(channels[0].properties["wf_increment"])
        time_off = float(channels[0].properties["wf_start_offset"])
    except KeyError:
        time_inc = 4e-6
        time_off = 0.0
        print(f"  {tdms_path.name}: no time metadata — using time_inc=4e-6 s")

    freq       = 1.0 / time_inc
    time_all_s = time_off + time_inc * np.arange(datasize)

    idx      = get_pol_ind(["0", "90", "45", "135"])
    c0_raw   = np.asarray(channels[idx[0]][:], dtype=float)
    c90_raw  = np.asarray(channels[idx[1]][:], dtype=float)
    c45_raw  = np.asarray(channels[idx[2]][:], dtype=float)
    c135_raw = np.asarray(channels[idx[3]][:], dtype=float)

    raw_data_all[tdms_path] = dict(
        c0=c0_raw, c90=c90_raw, c45=c45_raw, c135=c135_raw,
        time_all_s=time_all_s,
        freq=freq, time_inc=time_inc, time_off=time_off,
        datasize=datasize,
    )
    print(f"Loaded: {tdms_path.name}")
    print(f"  {datasize:,} pts  |  {freq:.0f} Hz  |  {time_all_s[-1]:.1f} s")

print(f"\nLoaded {len(raw_data_all)} / {len(tdms_file_list)} file(s).")


In [ ]:


# ── Global gain calibration ────────────────────────────────────────────────────
#
# Goal: find ONE gain vector a = [a_90, a_45, a_135, a_0] shared across all files
#       such that the balance condition holds everywhere:
#
#   a_0·c0 + a_90·c90 = a_45·c45 + a_135·c135
#
# We fix a_0 = 1 (reference channel) and solve for [a_90, a_45, a_135] by
# linear least squares across the concatenated data of all files:
#
#   A · x = b
#   [c90, -c45, -c135] · [a_90, a_45, a_135]^T = -c0
#
# This is the closed-form minimiser of the same cost function that the
# per-file optimizer uses — but solved jointly so every file contributes
# equally to determining a single shared set of coefficients.

rows_A = []
rows_b = []

for tdms_path, raw in raw_data_all.items():
    datasize = raw['datasize']
    time_inc = raw['time_inc']
    time_off = raw['time_off']

    # Time-window indices
    fit_s = (int((fit_start_s - time_off) / time_inc) if fit_start_s is not None else 0)
    fit_e = (int((fit_end_s   - time_off) / time_inc) if fit_end_s   is not None else datasize)
    fit_s = max(0, min(fit_s, datasize))
    fit_e = max(fit_s, min(fit_e, datasize))
    sl    = slice(fit_s, fit_e, dec_fit)

    # Raw channels (no offset — gain calibration uses raw data directly)
    c90  = raw['c90'][sl]
    c45  = raw['c45'][sl]
    c135 = raw['c135'][sl]
    c0   = raw['c0'][sl]

    rows_A.append(np.column_stack([ c90, -c45, -c135]))
    rows_b.append(-c0)
    print(f"  {tdms_path.name}: {len(c0):,} points added to fit")

A_mat = np.vstack(rows_A)
b_vec = np.concatenate(rows_b)

print(f"\nSolving {A_mat.shape[0]:,} equations × 3 unknowns …")
(a90, a45, a135), _, rank, sv = np.linalg.lstsq(A_mat, b_vec, rcond=None)

# a in physical stack order [a_90, a_45, a_135, a_0]
a_calibrated = [float(a90), float(a45), float(a135), 1.0]

print(f"\nCalibrated gain factors  (a_0 fixed = 1.0 as reference):")
print(f"  a_90  = {a_calibrated[0]:.6f}")
print(f"  a_45  = {a_calibrated[1]:.6f}")
print(f"  a_135 = {a_calibrated[2]:.6f}")
print(f"  a_0   = {a_calibrated[3]:.6f}  (fixed)")
print(f"\n  Matrix rank: {rank}   |   singular values: {sv}")


In [ ]:


# ── Verification: balance scatter ─────────────────────────────────────────────
# For each file plot (C0+C90) vs (C45+C135) scatter after gain correction.
# Points should lie on the y=x line when the calibration is correct.

a = a_calibrated   # [a_90, a_45, a_135, a_0]

n_files = len(raw_data_all)
ncols   = min(3, n_files)
nrows   = max(1, int(np.ceil(n_files / ncols)))

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(4.6 * ncols, 4.2 * nrows),
                          dpi=100, constrained_layout=True,
                          squeeze=False)
axes_flat = axes.flatten()

for ax_i, (tdms_path, raw) in enumerate(raw_data_all.items()):
    sl = slice(None, None, dec_plot)

    C0   = a[3] * raw['c0'][sl]
    C90  = a[0] * raw['c90'][sl]
    C45  = a[1] * raw['c45'][sl]
    C135 = a[2] * raw['c135'][sl]

    sum_x = C0  + C90
    sum_y = C45 + C135

    lim = float(np.nanpercentile(sum_x, 99.5))
    ax  = axes_flat[ax_i]
    ax.scatter(sum_x, sum_y, s=0.5, alpha=0.2, c="tab:blue", rasterized=True)
    ax.plot([0, lim], [0, lim], "k--", lw=0.8, label="y=x")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("C0 + C90", fontsize=8)
    ax.set_ylabel("C45 + C135", fontsize=8)
    ax.set_title(tdms_path.name, fontsize=7)
    ax.set_aspect("equal")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, frameon=False)

for j in range(n_files, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Balance verification: C0+C90 vs C45+C135  (after global gain calibration)", fontsize=9)
plt.show()


In [ ]:


# ── Verification: balance residual vs time ────────────────────────────────────
# For each file show (C0+C90) − (C45+C135) before and after calibration.
# After calibration the trace should be centred tightly around zero.

n_files = len(raw_data_all)
ncols   = 1
nrows   = n_files

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(12, 2.8 * nrows),
                          dpi=100, constrained_layout=True,
                          squeeze=False)

for ax_i, (tdms_path, raw) in enumerate(raw_data_all.items()):
    sl     = slice(None, None, dec_plot)
    time_p = raw['time_all_s'][sl]

    c0   = raw['c0'][sl]
    c90  = raw['c90'][sl]
    c45  = raw['c45'][sl]
    c135 = raw['c135'][sl]

    raw_res = (c0 + c90) - (c45 + c135)
    cor_res = (a[3]*c0 + a[0]*c90) - (a[1]*c45 + a[2]*c135)

    ax = axes[ax_i, 0]
    ax.plot(time_p, raw_res, lw=0.4, alpha=0.5, color="tab:grey",  label="before calibration")
    ax.plot(time_p, cor_res, lw=0.5, color="tab:blue",              label="after calibration")
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xlabel("Time (s)", fontsize=8)
    ax.set_ylabel("(C0+C90)−(C45+C135)", fontsize=8)
    ax.set_title(tdms_path.name, fontsize=8)
    ax.legend(fontsize=7, frameon=False)
    ax.grid(alpha=0.25)

plt.suptitle("Balance residual (C0+C90) − (C45+C135)  before / after gain calibration", fontsize=9)
plt.show()


In [ ]:


# ── Per-file residual RMS table ────────────────────────────────────────────────
# Compare the RMS of the balance residual before and after calibration.
# A good calibration should substantially reduce the RMS across all files.

print(f"  {'File':<50} {'RMS before':>12} {'RMS after':>12} {'ratio':>8}")
print("  " + "-" * 86)

for tdms_path, raw in raw_data_all.items():
    sl = slice(None, None, dec_plot)

    c0   = raw['c0'][sl]
    c90  = raw['c90'][sl]
    c45  = raw['c45'][sl]
    c135 = raw['c135'][sl]

    raw_res = (c0 + c90) - (c45 + c135)
    cor_res = (a[3]*c0 + a[0]*c90) - (a[1]*c45 + a[2]*c135)

    rms_before = float(np.sqrt(np.mean(raw_res**2)))
    rms_after  = float(np.sqrt(np.mean(cor_res**2)))
    ratio      = rms_after / rms_before if rms_before > 0 else float("nan")

    print(f"  {tdms_path.name:<50} {rms_before:>12.2f} {rms_after:>12.2f} {ratio:>8.4f}")


In [ ]:


# ── Results — copy these into Fourkas_processing.ipynb ────────────────────────

print("=" * 60)
print("Calibrated gain factors  [a_90, a_45, a_135, a_0]")
print("=" * 60)
for name, val in zip(["a_90", "a_45", "a_135", "a_0"], a_calibrated):
    print(f"  {name}  =  {val:.8f}")

print()
print("─── Paste into Fourkas_processing.ipynb config cell ───")
print(f"a_manual = {[round(v, 8) for v in a_calibrated]}")
print("         # ↑ [a_90, a_45, a_135, a_0]")
print()
print("─── Or as a dict for reference ────────────────────────")
print("a_manual = {")
for name, val in zip(["a_90", "a_45", "a_135", "a_0"], a_calibrated):
    print(f"    '{name}': {val:.8f},")
print("}")
